### Translating Recursion to WASM

This exercise comes with a modified Python P0 library: for testing purposes, the library function `read()` always returns `7`.

In [ ]:
def runpywasm(wasmfile):
  from pywasm.core import Machine, Runtime, FuncType, ValType

  # P0lib implementation in Python
  def write(_: Machine, args: list[int]) -> list[int]:
    print(args[0])
    return []

  def writeln(_: Machine, args: list[int]) -> list[int]:
    print()
    return []

  def read(_: Machine, args: list[int]) -> list[int]:
    return [7]  # [int(input())]

  # Create runtime
  runtime = Runtime()
  runtime.imports = {
    'P0lib': {
      'write': runtime.allocate_func_host(FuncType([ValType.i32()], []), write),
      'writeln': runtime.allocate_func_host(FuncType([], []), writeln),
      'read': runtime.allocate_func_host(FuncType([], [ValType.i32()]), read),
    }
  }
  # Create and run instance
  instance = runtime.instance_from_file(wasmfile)

The Fibonacci numbers can be computed in P0 by:

```Pascal
procedure fib(n: integer) → (r: integer)
  var a, b: integer
    if n ≤ 1 then r := n
    else
      a ← fib(n - 1); b ← fib(n - 2); r := a + b

program fibonacci
  var x: integer
    x ← read(); x ← fib(x); write(x)
```

Translate this program by hand to WebAssembly. The WebAssembly functions for `fib` and `fibonacci` should not use local or global variables, only the parameters, results, and the stack. That is, do _not_ follow the translation scheme of the P0 compiler, but write code that is hand-optimized for brevity. For this, you can use the fact that an `if` can push a result on the stack, provided that both branches do it. The syntax for this is:

```
        if (result i32)
            ... code that pushes i32 on the stack
        else
            ... code that pushes i32 on the stack
        end
```

In [ ]:
%%writefile fibonacci.wat
(module
(import "P0lib" "write" (func $write (param i32)))
(import "P0lib" "writeln" (func $writeln))
(import "P0lib" "read" (func $read (result i32)))
;;  procedure fib(n: integer) → (r: integer)
(func $fib (param $n i32) (result i32)
  ;;    if n ≤ 1 then r := n
  local.get $n
  i32.const 1
  i32.le_s
  if (result i32)
    local.get $n
  ;;    else a ← fib(n - 1); b ← fib(n - 2); r := a + b
  else
    local.get $n
    i32.const 1
    i32.sub
    call $fib
    local.get $n
    i32.const 2
    i32.sub
    call $fib
    i32.add
  end
)
;;  program fibonacci
(func $program
  ;;  var x: integer
  ;;    x ← read(); x ← fib(x); write(x)
  call $read
  call $fib
  call $write
)
(memory 1)
(start $program)
)

In [ ]:
!wat2wasm fibonacci.wat

In [ ]:
runpywasm('fibonacci.wasm')

You may test your program with:

In [ ]:
%%capture output
runpywasm('fibonacci.wasm')

In [ ]:
assert str(output) == '13\n'

The cells below are for grading purposes only; please ignore them.

In [ ]:
%%capture output